# audio

> Audio-JEPA context encoder, as `VisionTransformer` runs it at inference.
>
> Dropout and stochastic depth are the identity. Blocks are pre-LayerNorm
> ViT blocks. Positions are a fixed 2D sine-cosine table.


In [ ]:
#| default_exp audio


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import numpy as np
from max.experimental.nn import LayerNorm, Module, ModuleList
from max.experimental.tensor import Tensor

from nada.config import AudioJEPAConfig
from nada.layers import Attention, Conv2d, MLP


Positions copy `get_2d_sincos_pos_embed` from the training repo. `meshgrid` is called with width first, then the first half of the vector encodes that axis and the second half encodes height.


In [ ]:
#| export
def sincos_pos_embed(embed_dim: int, grid_h: int, grid_w: int) -> np.ndarray:
    """2D sine-cosine table of shape `(1, grid_h * grid_w, embed_dim)`."""
    grid_h_idx = np.arange(grid_h, dtype=np.float64)
    grid_w_idx = np.arange(grid_w, dtype=np.float64)
    grid = np.stack(np.meshgrid(grid_w_idx, grid_h_idx), axis=0)
    grid = grid.reshape(2, -1)

    def encode(dim: int, pos: np.ndarray) -> np.ndarray:
        omega = np.arange(dim // 2, dtype=np.float64)
        omega /= dim / 2.0
        omega = 1.0 / (10000 ** omega)
        out = np.outer(pos, omega)
        return np.concatenate([np.sin(out), np.cos(out)], axis=1)

    pos = np.concatenate([encode(embed_dim // 2, grid[0]), encode(embed_dim // 2, grid[1])], axis=1)
    return pos.astype(np.float32)[None]


In [ ]:
#| export
class PatchEmbed(Module):
    """`(batch, time, mel, 1)` in, `(batch, patches, embed_dim)` out."""

    def __init__(self, config: AudioJEPAConfig) -> None:
        self.grid_time = config.grid_time
        self.grid_freq = config.grid_freq
        self.proj = Conv2d(
            config.in_chans,
            config.embed_dim,
            config.patch_size,
            stride=config.patch_size,
            bias=True,
        )

    def forward(self, spectrogram: Tensor) -> Tensor:
        hidden = self.proj(spectrogram)
        batch = hidden.shape[0]
        return hidden.reshape([batch, self.grid_time * self.grid_freq, hidden.shape[3]])


In [ ]:
#| export
class Block(Module):
    """Pre-LN block: norm, attention, add, norm, MLP, add."""

    def __init__(self, config: AudioJEPAConfig) -> None:
        self.norm1 = LayerNorm(config.embed_dim, eps=config.layer_norm_eps)
        self.attn = Attention(config.embed_dim, config.num_heads, config.qkv_bias)
        self.norm2 = LayerNorm(config.embed_dim, eps=config.layer_norm_eps)
        self.mlp = MLP(config.embed_dim, config.mlp_hidden)

    def forward(self, hidden: Tensor) -> Tensor:
        hidden = hidden + self.attn(self.norm1(hidden))
        return hidden + self.mlp(self.norm2(hidden))


In [ ]:
#| export
class VisionTransformer(Module):
    """Spectrogram in, one `embed_dim` vector per patch out."""

    def __init__(self, config: AudioJEPAConfig) -> None:
        self.config = config
        self.patch_embed = PatchEmbed(config)
        self.pos_embed = Tensor(sincos_pos_embed(config.embed_dim, config.grid_time, config.grid_freq))
        self.blocks = ModuleList([Block(config) for _ in range(config.depth)])
        self.norm = LayerNorm(config.embed_dim, eps=config.layer_norm_eps)

    def forward(self, spectrogram: Tensor) -> Tensor:
        hidden = self.patch_embed(spectrogram) + self.pos_embed
        for block in self.blocks:
            hidden = block(hidden)
        return self.norm(hidden)


In [ ]:
table = sincos_pos_embed(8, 2, 3)
assert table.shape == (1, 6, 8)
assert np.isfinite(table).all()
